# 08 · Memory: Conversations 🧠

Every `responses.create` starts with a blank mind — unless we tell OpenAI which chat to continue.
At the end, we set up your bot's memory **with your PDF notes inside**.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-6-luna"
print("Ready!")

In [ ]:
bot = "You are SG Data Bot, a helpful assistant for life in Singapore. Keep replies short."

## 1. Without memory 🐟

In [ ]:
client.responses.create(model=MODEL, instructions=bot, input="Hi, I'm Ahmad. I live in Bedok and I cycle to work.")

response = client.responses.create(model=MODEL, instructions=bot, input="Where do I live?")
print(response.output_text)

The bot forgot already!

## 2. Memory way #1: `previous_response_id`
You used this in notebook 06. It says *"continue from this response"*.

In [ ]:
first = client.responses.create(model=MODEL, instructions=bot, input="Hi, I'm Ahmad. I live in Bedok and I cycle to work.")

second = client.responses.create(model=MODEL, instructions=bot, previous_response_id=first.id, input="Where do I live?")
print(second.output_text)

It works, but we must keep track of the **latest** response ID after every message. If the bot restarts, the ID is lost.

## 3. Memory way #2: a conversation 💾
A **conversation** is a chat history that OpenAI keeps for us. It has **one ID that never changes** — easy to save!

In [ ]:
conversation = client.conversations.create()

client.responses.create(model=MODEL, instructions=bot, conversation=conversation.id,
                        input="Hi, I'm Ahmad. I live in Bedok and I cycle to work.")

response = client.responses.create(model=MODEL, instructions=bot, conversation=conversation.id,
                                   input="Where do I live?")
print(response.output_text)

## 4. Peek inside the conversation

In [ ]:
items = client.conversations.items.list(conversation.id, order="asc")

for item in items.data:
    if item.type == "message":
        print(f"{item.role:>9}: {item.content[0].text}")

## 5. Set up your bot's memory 🗂️
Your bot (`main.py`) needs two IDs:
- **a conversation** that already contains your PDF notes, so the bot knows them from the start
- **the HDB file**, for the python tool

In [ ]:
# 1. a new conversation for the bot
bot_conversation = client.conversations.create()

# 2. send your PDF into it once - the bot will remember it
notes_file = client.files.create(file=open("my_flat_notes.pdf", "rb"), purpose="user_data")
response = client.responses.create(
    model=MODEL,
    instructions=bot,
    conversation=bot_conversation.id,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Here are my flat-hunting notes. Remember them for later."},
            {"type": "input_file", "file_id": notes_file.id},
        ],
    }],
)
print(response.output_text)

# 3. upload the HDB prices for the python tool
hdb_file = client.files.create(file=open("hdb_resale_prices.csv", "rb"), purpose="user_data")

In [ ]:
# 4. save both IDs in your .env file
with open(".env", "a") as f:
    f.write(f"\nOPENAI_CONVERSATION_ID={bot_conversation.id}\n")
    f.write(f"OPENAI_HDB_FILE_ID={hdb_file.id}\n")

print("Saved! Open .env to check.")

> 👤 We're keeping it simple: **one bot, one person**. Everyone who chats with your bot shares this one conversation.

## ✏️ Exercises
1. Count how many messages are in `bot_conversation`.
2. Restart the kernel (⟳), run the setup cell, then chat using `conversation=os.environ["OPENAI_CONVERSATION_ID"]`. Ask *"What's my budget?"*
3. Use your **own** PDF instead (your CV, a guide, a menu...) — drag it into the file list first.

In [ ]:
# your code here